# ANN vs Transformer — model comparison

This notebook presents the results of `compare_models.py`. Each architecture was trained with 5 random seeds on identical data, windows, splits, training target (the hour-to-hour change) and loss, so the only difference between them is architecture.

It reads the saved results and does not retrain anything. To regenerate them, run `.venv/bin/python compare_models.py` (about 70 minutes on CPU), or run the optional cell below. Finished runs are cached in `results/comparison_runs/`, so only missing ones are trained.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
for p in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

RESULTS_DIR = PROJECT_ROOT / "results"
FIG_DIR = PROJECT_ROOT / "figures"


def show(name: str, width: int = 820) -> None:
    display(Image(filename=str(FIG_DIR / f"{name}.png"), width=width))

### Optional: regenerate the results

In [ ]:
RERUN = False  # set True to (re)run compare_models.py from this notebook

if RERUN:
    import compare_models
    compare_models.main()

## 1. Headline results (test set)

`mean` and `std` are taken across the 5 seeds. `ensemble` scores the average of the 5 seeds' forecasts. Lower is better for MAE, RMSE and MAPE; higher is better for R².

In [ ]:
summary = pd.read_csv(RESULTS_DIR / "comparison_summary.csv")
table = summary.pivot_table(index=["target", "metric"], columns="model",
                            values=["mean", "std", "ensemble"])
table.columns = [f"{model} {stat}" for stat, model in table.columns]
persistence = summary.drop_duplicates(["target", "metric"]).set_index(["target", "metric"])["persistence"]
table["Persistence"] = persistence
order = [c for m in ["ANN", "Transformer"] for c in (f"{m} mean", f"{m} std", f"{m} ensemble")]
table[order + ["Persistence"]].style.format("{:,.3f}")

## 2. Is the difference real? Diebold–Mariano tests

The DM test checks whether two forecasts are equally accurate over the test set, using the ensemble forecasts. The variance uses Newey–West with 24 lags, because hourly errors are autocorrelated. A **negative** statistic means the first model is more accurate. `|error|` compares absolute errors (MAE) and `error²` compares squared errors (MSE/RMSE).

In [ ]:
dm = json.loads((RESULTS_DIR / "comparison_summary.json").read_text())
rows = []
for target, s in dm.items():
    for pair, tests in s["diebold_mariano"].items():
        rows.append({
            "target": target, "comparison": pair,
            "DM |error|": tests["absolute_error"]["statistic"],
            "p |error|": tests["absolute_error"]["p_value"],
            "DM error²": tests["squared_error"]["statistic"],
            "p error²": tests["squared_error"]["p_value"],
        })
pd.DataFrame(rows).set_index(["target", "comparison"]).style.format(
    {"DM |error|": "{:+.2f}", "DM error²": "{:+.2f}", "p |error|": "{:.3g}", "p error²": "{:.3g}"})

## 3. Per-seed results

These are the individual runs behind the means. `val_*` columns are on the validation set, which was used for early stopping and for choosing settings. The other metric columns are on the test set.

In [ ]:
runs = pd.read_csv(RESULTS_DIR / "comparison_runs.csv")
runs.set_index(["target", "model", "seed"]).style.format(precision=3)

## 4. Figures

In [ ]:
for name in ["13_model_comparison", "14_co2_comparison_timeseries",
             "14_nox_comparison_timeseries", "15_error_by_regime"]:
    show(name)

### Reading the figures

- **Figure 13:** each dot is one seed. If one model's range sits entirely below the other's, the ranking does not depend on random initialisation.
- **Figure 14:** the most volatile 4 days of the test set. On large ramps, both models mostly track the actual curve about one hour late, much like persistence. One hour ahead, neither can anticipate when a ramp starts from past data alone, only partly how it continues.
- **Figure 15:** in the steadiest quarter of hours, persistence is nearly unbeatable. The models earn their keep in the largest-ramp quarter, and that is where their RMSE advantage comes from.